# Machine Doctor — Deep Learning Add-on
## Step 5: Rigorous Evaluation — Leave-One-Load-Out Leakage Check

**Why this step exists:** Step 4 got 100% test accuracy with unstable test-loss swings — a classic symptom of *condition leakage*. We split by file, which guarantees no identical window repeats between train/test, but files at the **same load** (different fault severities) can still share subtle recording-session fingerprints (motor hum, sensor calibration) that have nothing to do with the actual fault physics. A network can "cheat" by learning to recognize the session instead of the fault.

**The real test:** train on 3 of the 4 load conditions (0/1/2/3 hp), then test on the load the model has *never seen at all*. We repeat this once per load (leave-one-load-out), so every single window in the dataset gets predicted exactly once, by a model that never trained on its load. Pooling all of those predictions gives us a genuinely honest confusion matrix and per-class accuracy — no leakage possible, because train and test never share a load.

If accuracy stays high here, Step 4's result was real. If it drops a lot, we just caught the leakage before shipping anything.

In [ ]:
!pip install -q kagglehub
import kagglehub, os, glob, re
import numpy as np
import scipy.io
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

dataset_path = kagglehub.dataset_download("esraakhaled299/cwru-data")
print("Dataset at:", dataset_path)

In [ ]:
# Same class labeling as Step 2, but this time we ALSO parse the load
# condition (0/1/2/3 hp) out of the filename -- CWRU filenames end in
# '_<load>.mat', e.g. 'OR007@6_0.mat', 'IR014_2.mat', 'Normal_3.mat'.
# This load digit is exactly the thing Step 2's split ignored.
CLASS_NAMES = ["Normal", "Ball", "InnerRace", "OuterRace"]

# NOTE: Normal baseline files live in their own 'Normal/Load_X/' folder,
# separate from the 12k_DE fault folder -- a plain '12k' substring filter
# silently drops them. Match on the actual folder structure instead.
all_files = glob.glob(os.path.join(dataset_path, "**", "*.mat"), recursive=True)
de_files = [f for f in all_files if "12k_DE" in f or f"{os.sep}Normal{os.sep}" in f]
if not de_files:
    de_files = all_files

LOAD_RE = re.compile(r"_(\d)\.mat$")

labeled_files = []  # (filepath, class_name, load)
skipped = []
for f in de_files:
    class_name = None
    for c in CLASS_NAMES:
        if c in f:
            class_name = c
            break
    m = LOAD_RE.search(os.path.basename(f))
    if class_name is not None and m is not None:
        labeled_files.append((f, class_name, int(m.group(1))))
    else:
        skipped.append(f)

print(f"Labeled {len(labeled_files)} files with both class + load")
if skipped:
    print(f"WARNING: {len(skipped)} files couldn't be parsed for load and were skipped, e.g.:")
    for f in skipped[:5]:
        print("  ", os.path.basename(f))

print("\nFile counts by class x load:")
loads = sorted(set(l for _, _, l in labeled_files))
header = "Class".ljust(12) + "".join(f"load {l}".rjust(9) for l in loads)
print(header)
for c in CLASS_NAMES:
    row = c.ljust(12)
    for l in loads:
        count = sum(1 for _, cc, ll in labeled_files if cc == c and ll == l)
        row += str(count).rjust(9)
    print(row)

assert len(skipped) == 0, "Fix the filename parsing before continuing -- we need every file's load."
assert set(loads) == {0, 1, 2, 3}, f"Expected loads 0-3, got {loads}"

In [ ]:
# Same windowing logic as Step 2 (1024-sample windows, 50% stride overlap,
# per-window normalization) -- but now windows carry their load along too,
# so we can split by load instead of by file.
WINDOW_SIZE = 1024
STRIDE = 512

def load_de_signal(filepath):
    mat = scipy.io.loadmat(filepath)
    key = [k for k in mat.keys() if "DE_time" in k][0]
    return mat[key].flatten()

def make_windows(file_label_list):
    X, y = [], []
    for filepath, class_name, _load in file_label_list:
        signal = load_de_signal(filepath)
        for start in range(0, len(signal) - WINDOW_SIZE, STRIDE):
            window = signal[start:start + WINDOW_SIZE]
            window = (window - window.mean()) / (window.std() + 1e-8)
            X.append(window)
            y.append(CLASS_NAMES.index(class_name))
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

In [ ]:
# Same CNN as Steps 3-4, wrapped in a factory function so we can train a
# fresh, untrained copy for each held-out load (reusing weights across
# folds would itself be a leak).
class VibrationCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=64, stride=2, padding=32),
            nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=32, stride=2, padding=16),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=16, stride=2, padding=8),
            nn.BatchNorm1d(64), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64, 32), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def make_model():
    return VibrationCNN(num_classes=len(CLASS_NAMES)).to(device)

In [ ]:
# One training run for one held-out load. Trains on the other 3 loads,
# evaluates only on the held-out one, and returns predictions so we can
# pool them into one big honest confusion matrix afterward.
def to_loader(X, y, batch_size=64, shuffle=False):
    X_t = torch.tensor(X).unsqueeze(1)
    y_t = torch.tensor(y)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

def run_fold(held_out_load, epochs=15, verbose=True):
    train_files = [t for t in labeled_files if t[2] != held_out_load]
    test_files  = [t for t in labeled_files if t[2] == held_out_load]

    X_train, y_train = make_windows(train_files)
    X_test,  y_test  = make_windows(test_files)

    train_loader = to_loader(X_train, y_train, shuffle=True)
    test_loader  = to_loader(X_test,  y_test,  shuffle=False)

    class_counts = np.array([max(1, np.sum(y_train == i)) for i in range(len(CLASS_NAMES))])
    class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
    class_weights = (class_weights / class_weights.sum() * len(CLASS_NAMES)).to(device)

    model = make_model()
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

    # Final eval on the held-out load
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch).argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
            all_true.append(y_batch.numpy())
    all_preds = np.concatenate(all_preds)
    all_true = np.concatenate(all_true)
    acc = (all_preds == all_true).mean()

    if verbose:
        print(f"Held-out load {held_out_load}: trained on {len(X_train)} windows "
              f"(loads {sorted(set(l for _,_,l in train_files))}), "
              f"tested on {len(X_test)} windows -> accuracy = {acc:.3f}")

    return all_true, all_preds, acc

In [ ]:
# Run all 4 folds. Every window in the whole dataset ends up predicted
# exactly once, by a model that never saw its load during training.
# This is the leakage-proof number -- pooled together it's the real
# answer to "does this model actually understand bearing faults."
per_load_acc = {}
pooled_true, pooled_preds = [], []

for held_out_load in sorted(loads):
    y_true, y_pred, acc = run_fold(held_out_load, epochs=15)
    per_load_acc[held_out_load] = acc
    pooled_true.append(y_true)
    pooled_preds.append(y_pred)

pooled_true = np.concatenate(pooled_true)
pooled_preds = np.concatenate(pooled_preds)
overall_acc = (pooled_true == pooled_preds).mean()

print("\nPer-load held-out accuracy:")
for l, a in per_load_acc.items():
    print(f"  load {l}: {a:.3f}")
print(f"\nOverall leave-one-load-out accuracy (every window, honest): {overall_acc:.3f}")
print(f"Compare to Step 4's file-based-split accuracy: ~1.000")

In [ ]:
# Confusion matrix + per-class accuracy on the FULL pooled, leakage-proof
# predictions -- this is the number that actually matters.
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(pooled_true, pooled_preds, labels=range(len(CLASS_NAMES)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Leave-one-load-out confusion matrix (leakage-proof)")
plt.tight_layout()
plt.show()

print("\nPer-class accuracy (leakage-proof):")
for i, name in enumerate(CLASS_NAMES):
    row = cm[i]
    class_acc = row[i] / row.sum() if row.sum() > 0 else float("nan")
    print(f"  {name}: {class_acc:.3f}  ({row[i]}/{row.sum()} correct)")

### How to read the result

- **If overall accuracy above is still high (roughly 90%+)** and reasonably close across all four per-load rows: Step 4's near-100% wasn't a fluke of leakage — the model is genuinely picking up fault physics that generalize across load/speed conditions. Good news.
- **If it drops noticeably (say, into the 60-80% range) or one particular load is much worse than the others**: that gap *is* the leakage Step 4 was hiding. The file-based split let the model partially learn to recognize recording sessions rather than faults, and this leave-one-load-out number is the trustworthy one to report/use going forward.
- Either way, this number — not Step 4's — is the one to quote as "real-world accuracy," since it's the only one where train and test never share a load.